# Candidate Generation: Neural Collaborative Filtering (NCF)

Personalised neural baseline that scores each request with an **MLP over concatenated user and article embeddings** rather than a dot product, item co-occurrence or popularity.

Protocol:

- Build positives from session clicks in the full `labels_train` pool and negatives from exposed non-clicks in the same slate.
- Redraw up to four exposed negatives per positive from the same slate on every epoch.
- Train BPR with AdamW, selecting the epoch by NDCG on a chronological 90% cut of the training period.
- Refit on the full training period for the selected number of epochs.
- Score each request by concatenating the static user and article embeddings and passing them through the MLP over articles with `available_from <= timestamp`, excluding the request's `full_history_rows`.
- Break ties by training exposure popularity descending, then ascending `news_code`; backfill short lists from popularity.
- Write the standard `impression_uid, cutoff, news_code` table and evaluate it with `cg_evaluation.py`.

The model is static: it is fitted once on the training split and never updated with target-period interactions. The embedding size `EMBED_DIM = 256`, the MLP shape `MLP_HIDDEN = (256, 128, 64)` and the 500-candidate output are fixed in this notebook.


## 0. Configure

**This is the only cell you need to edit.** Set `REUSE_ARTIFACTS = False` to retrain the model
and rebuild every artifact. Candidate lists are always regenerated for `TARGET_SPLIT`, so a new
validation or test split is served normally.


In [ ]:
# ---- Where to run ----------------------------------------------------------------------------
# "colab" = Google Colab with Google Drive, "local" = this computer.
ENVIRONMENT = "colab"  # "local" or "colab"

# ---- Paths -------------------------------------------------------------------------------------
# Data path: the prepared data. It must contain
# 1_Data_Preparation/processed/, 3_Representations/artifacts/ and
# 4_Feature_Engineering/artifacts/prequential-all/.
# On Colab, the zipped data in <submission>/2.Feature_Engineering is unpacked into the data
# path on the first run (a few minutes); later runs reuse it.
COLAB_DATA_PATH = None  # used on Colab; None = the project folder (4.Code)
LOCAL_DATA_PATH = None  # used locally; None = the project folder

# ---- Experiment ------------------------------------------------------------------------------
# Split to evaluate: "test" for the final results, or "validation".
TARGET_SPLIT = "validation"

# True: load the saved model checkpoint and skip training. Serving still runs, so candidate
# lists are always regenerated for TARGET_SPLIT.
# False: retrain the model from scratch.
# Baselines without a model reuse any cached intermediate state, but their candidate lists are
# still regenerated.
REUSE_ARTIFACTS = True

# ---- Reused artifacts when REUSE_ARTIFACTS=True -----------------------------------------------
# Loaded checkpoint, so training is skipped:
#   artifacts/ncf_bpr/model.pt
# Candidate lists are always regenerated.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn


if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    CODE_ROOT = Path("/content/drive/MyDrive/submission/4.Code")
else:
    CODE_ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
    )
ROOT = CODE_ROOT
DATA_PATH = COLAB_DATA_PATH if ENVIRONMENT == "colab" else LOCAL_DATA_PATH
DATA_ROOT = (
    Path(
        DATA_PATH
        or os.environ.get("NEWSREC_DATA_ROOT")
        or os.environ.get("NEWSREC_ROOT")
        or CODE_ROOT
    )
    .expanduser()
    .resolve()
)
REQUIRED_DATA = [
    "1_Data_Preparation/processed",
    "3_Representations/artifacts",
    "4_Feature_Engineering/artifacts/prequential-all",
]
SUBMISSION_ROOT = CODE_ROOT.parent if ENVIRONMENT == "colab" else None
BUNDLE_DIR = SUBMISSION_ROOT / "2.Feature_Engineering" if SUBMISSION_ROOT else None
if BUNDLE_DIR is not None and BUNDLE_DIR.is_dir():
    import zipfile

    for archive in sorted(BUNDLE_DIR.glob("*.zip")):
        marker = DATA_ROOT / f".unpacked_{archive.stem}"
        if not marker.exists():
            print(f"Unpacking {archive.name} into {DATA_ROOT} (first run only) ...")
            with zipfile.ZipFile(archive) as bundle:
                bundle.extractall(DATA_ROOT)
            marker.touch()

missing_data = [
    str(DATA_ROOT / sub) for sub in REQUIRED_DATA if not (DATA_ROOT / sub).exists()
]
if missing_data:
    raise FileNotFoundError(
        f"{ENVIRONMENT.upper()}_DATA_PATH is missing these folders:\n"
        + "\n".join(missing_data)
    )
REPLAY_ROOT = DATA_ROOT
CG_DIR = CODE_ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(CODE_ROOT / "src"))

from cf_models import load_context, resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)

from newsrec.common import Settings

PROCESSED = DATA_ROOT / "1_Data_Preparation" / "processed"
FEATURES = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"
ARTIFACTS = (
    CODE_ROOT.parent / "3.Model_Artifacts" / "CG"
    if ENVIRONMENT == "colab"
    else DATA_ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
)
RESULTS = DATA_ROOT / "results"

RUN_LOSSES = ("bpr",)
LOSS_CONFIGS = {
    loss: {
        "model_name": f"ncf_{loss}",
        "artifact_dir": ARTIFACTS / f"ncf_{loss}",
        "recommendations_path": ARTIFACTS
        / f"ncf_{loss}"
        / f"recommendations_{TARGET_SPLIT}_top500.parquet",
        "result_path": RESULTS / f"candidate_generation_ncf_{loss}_{TARGET_SPLIT}.csv",
    }
    for loss in RUN_LOSSES
}
for config in LOSS_CONFIGS.values():
    config["artifact_dir"].mkdir(parents=True, exist_ok=True)

M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
EMBED_DIM = 256
MLP_HIDDEN = (256, 128, 64)
SCORE_PAIRS = 524_288
NUM_EPOCHS = 50
BATCH_SIZE = 32_768
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6
SEED = 4222
PATIENCE = 3
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
DEVICE = resolve_device("auto")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Repository:", ROOT)
print("Model:", LOSS_CONFIGS[RUN_LOSSES[0]]["model_name"])
print("Device:", DEVICE)
print("Torch:", torch.__version__)

## 2. Data ingestion

Positives are session clicks from the full `labels_train` pool, so history-only exposures are not treated as positives. Each training epoch redraws up to four exposed non-clicks from the same slate; the sampler is restricted to the chronological fit window for tuning and then applied to all rows for the refit. Tuning slates are sampled deterministically from `candidates_train`, NDCG uses the deepest slate length as `k`, and article codes are mapped to `article_row` through `article_index.parquet`.


In [2]:
context = load_context(DATA_ROOT)

In [3]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code"],
).sort("article_row")

run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
if run_manifest.get("training_candidate_pool") != "complete":
    raise ValueError(
        "Per-epoch sampling needs the complete training pool. "
        "Rebuild stage 4 with the default training_negative_sampling='epoch'."
    )
CFG = Settings(**run_manifest["config"])

TRAINING_ROWS = (
    pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
    .select("impression_uid", "news_code", "clicked")
    .to_pandas()
)
if (
    TRAINING_ROWS["clicked"].isna().any()
    or not TRAINING_ROWS["clicked"].isin([0, 1]).all()
):
    raise ValueError("Training rows must be uniquely binary-labelled")

train_profiles = pl.read_parquet(
    sorted((FEATURES / "request_profiles_train").glob("*.parquet")),
    columns=["impression_uid", "user_code", "timestamp"],
)

positive_uids = np.unique(
    TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
)
positive_profiles = train_profiles.filter(pl.col("impression_uid").is_in(positive_uids))
cutoff_ns = chronological_cutoff(
    positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
)
fit_profiles = chronological_mask(train_profiles["timestamp"].to_numpy(), cutoff_ns)
fit_uids = set(
    train_profiles.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
)
FIT_ROWS = TRAINING_ROWS[TRAINING_ROWS["impression_uid"].isin(fit_uids)].reset_index(
    drop=True
)
ALL_ROWS = TRAINING_ROWS
n_groups = int(TRAINING_ROWS["clicked"].sum())
EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


def sample_epoch(epoch, *, fit_only):
    """Draw exposed negatives for one epoch and build NCF training arrays."""
    groups, _ = EPOCH_SAMPLER.draw(int(epoch), uids=fit_uids if fit_only else None)
    groups = (
        pl.from_pandas(groups)
        .join(train_profiles, on="impression_uid", how="left")
        .with_row_index("group_row")
        .join(
            article_index.select(
                pl.col("news_code"),
                pl.col("article_row").alias("pos_row"),
            ),
            left_on="positive_news_code",
            right_on="news_code",
            how="left",
        )
    )
    if groups["user_code"].null_count() or groups["pos_row"].null_count():
        raise ValueError("Sampled groups are missing training profiles or article rows")
    negative_long = (
        groups.select("group_row", "negative_news_codes")
        .explode("negative_news_codes", empty_as_null=True)
        .rename({"negative_news_codes": "negative_news_code"})
        .drop_nulls("negative_news_code")
        .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
        .join(
            article_index.select(
                pl.col("news_code").alias("negative_news_code"),
                pl.col("article_row").alias("neg_row"),
            ),
            on="negative_news_code",
            how="left",
        )
        .sort(["group_row", "neg_pos"])
    )
    if negative_long["neg_row"].null_count():
        raise ValueError("Sampled negatives are missing article rows")
    n_epoch_groups = groups.height
    NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
    if n_epoch_groups and NEG_MAX == 0:
        raise ValueError("Sampled training rows contain no exposed negatives")
    NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
    group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
    NEG_MATRIX[group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)] = (
        negative_long["neg_row"].to_numpy().astype(np.int64)
    )
    NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
    return (
        groups,
        groups["pos_row"].to_numpy().astype(np.int64),
        NEG_MATRIX,
        NEG_LEN,
    )


TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    train_profiles.select("impression_uid", "user_code"),
    max_impressions=None,
    seed=SEED + 1,
)
TUNE_UIDS = tune["impression_uid"].to_numpy()
TUNE_USERS = tune["user_code"].to_numpy().astype(np.int64)
TUNE_ITEMS = tune["article_row"].to_numpy().astype(np.int64)
TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)

print(f"Users: {context.n_users:,}")
print(f"Articles: {context.n_items:,}")
print(f"Training rows: {len(TRAINING_ROWS):,}")
print(f"Training positives: {n_groups:,}")
print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
print(f"Sampler negatives per positive: up to {CFG.negative_count}")
print(f"Tuning cutoff: {np.datetime64(cutoff_ns, 'ns')}")
print(f"Tuning rows: {tune.height:,}")
print(f"Tuning NDCG k (max training clicks per impression): {TUNE_NDCG_K}")

Users: 50,001
Articles: 51,283
Training rows: 3,389,342
Training positives: 141,558
Chronological fit rows: 3,046,331 / 3,389,342
Sampler negatives per positive: up to 4
Tuning cutoff: 2019-11-12T13:50:10.000000000
Tuning rows: 343,011
Tuning NDCG k (max training clicks per impression): 31


## 3. Model

NCF has exactly two embedding tables: a static `user_code` embedding and an `article_row` embedding, both of width `EMBED_DIM = 256` and initialised from `N(0, 0.01^2)`. The two vectors are concatenated into a 512-d input for an MLP with hidden sizes `256 -> 128 -> 64` and GELU activations; the scalar output is the ranking score. There is no dot product, bias term or calibrated probability, and the padding row `0` is zeroed and never trained.

Architecture:

```text
                 +-----------------------------------+
user_code ------>| user_embedding         (N_u x 256) |----> u --+
                 +-----------------------------------+           |
                                                                 +--> concat (512) --> MLP --> score
                 +-----------------------------------+           |
article_row ---->| item_embedding         (N_i x 256) |----> v --+
                 +-----------------------------------+

MLP: 512 -> 256 -> GELU -> 128 -> GELU -> 64 -> GELU -> 1

BPR objective (one positive vs up to 4 exposed negatives per step):

s_pos = MLP([u, v_clicked])
s_neg = MLP([u, v_shown_not_clicked])
loss  = -mean( log sigmoid(s_pos - s_neg) )
```


In [4]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, hidden_dims):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

        layers = []
        in_dim = 2 * embed_dim
        for hidden_dim in hidden_dims:
            layers += [nn.Linear(in_dim, hidden_dim), nn.GELU()]
            in_dim = hidden_dim
        self.mlp = nn.Sequential(*layers, nn.Linear(in_dim, 1))

    def score_embeddings(self, user_vectors, item_vectors):
        return self.mlp(torch.cat([user_vectors, item_vectors], dim=-1)).squeeze(-1)

    def forward(self, user_ids, item_ids):
        return self.score_embeddings(
            self.user_embedding(user_ids),
            self.item_embedding(item_ids),
        )

## 4. Training

Every epoch rebuilds the negative matrix from `EpochNegativeSampler`, then optimises BPR on `positive_score - negative_score` with AdamW. The epoch with the best tuning NDCG is kept (patience `7`), its checkpoint is restored, and a fresh model is refit for that many epochs on all training rows. The checkpoint, training log and configuration are written under `artifacts/ncf_<loss>/`.


In [5]:
def train_epoch(model, optimizer, user_ids, item_ids, neg_matrix, neg_len, rng):
    model.train()
    order = rng.permutation(len(user_ids))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(user_ids[batch], dtype=torch.long, device=DEVICE)
        positives = torch.as_tensor(item_ids[batch], dtype=torch.long, device=DEVICE)
        negatives = torch.as_tensor(neg_matrix[batch], dtype=torch.long, device=DEVICE)
        neg_lens = torch.as_tensor(neg_len[batch], dtype=torch.long, device=DEVICE)
        neg_mask = (
            torch.arange(negatives.shape[1], device=DEVICE)[None, :] < neg_lens[:, None]
        )
        users_expanded = users[:, None].expand(-1, negatives.shape[1])
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model(users, positives)
        negative_scores = model(
            users_expanded.reshape(-1), negatives.reshape(-1)
        ).reshape(len(batch), negatives.shape[1])
        if LOSS == "bpr":
            loss = (
                -(
                    F.logsigmoid(positive_scores[:, None] - negative_scores) * neg_mask
                ).sum()
                / neg_mask.sum()
            )
        else:
            loss = F.binary_cross_entropy_with_logits(
                torch.cat([positive_scores, negative_scores[neg_mask]]),
                torch.cat(
                    [
                        torch.ones_like(positive_scores),
                        torch.zeros_like(negative_scores[neg_mask]),
                    ]
                ),
            )
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)

In [6]:
def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), 200_000):
            end = min(start + 200_000, len(TUNE_CLICKED))
            users = torch.as_tensor(
                TUNE_USERS[start:end], dtype=torch.long, device=DEVICE
            )
            items = torch.as_tensor(
                TUNE_ITEMS[start:end], dtype=torch.long, device=DEVICE
            )
            scores[start:end] = model(users, items).cpu().numpy()
    _, ndcg = mean_slate_metrics(TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K)
    return ndcg

In [ ]:
TRAINED_MODELS = {}
for loss in RUN_LOSSES:
    LOSS = loss
    config = LOSS_CONFIGS[loss]
    print(f"[{LOSS}] Training {config['model_name']}")

    model_path = config["artifact_dir"] / "model.pt"
    expected_config = {
        "n_users": context.n_users,
        "n_items": context.n_items,
        "embed_dim": EMBED_DIM,
        "mlp_hidden": list(MLP_HIDDEN),
        "objective": LOSS,
    }
    if REUSE_ARTIFACTS and model_path.exists():
        checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
        saved_config = checkpoint.get("config", {})
        if all(saved_config.get(key) == value for key, value in expected_config.items()):
            model = NCF(context.n_users, context.n_items, EMBED_DIM, MLP_HIDDEN).to(
                DEVICE
            )
            model.load_state_dict(checkpoint["state_dict"])
            model.eval()
            TRAINED_MODELS[LOSS] = model
            print(f"[{LOSS}] Loaded model: {model_path}")
            continue
        print(f"[{LOSS}] Checkpoint config mismatch; retraining")

    model = NCF(context.n_users, context.n_items, EMBED_DIM, MLP_HIDDEN).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 1)
    training_log = []
    stopper = EarlyStopper(patience=PATIENCE)

    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        groups, pos_rows, neg_matrix, neg_len = sample_epoch(epoch - 1, fit_only=True)
        loss = train_epoch(
            model,
            optimizer,
            groups["user_code"].to_numpy().astype(np.int64),
            pos_rows,
            neg_matrix,
            neg_len,
            rng,
        )
        ndcg = tune_ndcg(model)
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "tune",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": ndcg,
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"epoch {epoch:02d} loss={loss:.6f} "
            f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
        )
        if stopper.step(ndcg, model, epoch):
            print("Early stopping")
            break

    stopper.restore(model)
    print(
        f"Best epoch: {stopper.best_epoch} "
        f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
    )

    final_model = NCF(context.n_users, context.n_items, EMBED_DIM, MLP_HIDDEN).to(
        DEVICE
    )
    optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 2)

    for epoch in range(1, stopper.best_epoch + 1):
        started = time.time()
        groups, pos_rows, neg_matrix, neg_len = sample_epoch(epoch - 1, fit_only=False)
        loss = train_epoch(
            final_model,
            optimizer,
            groups["user_code"].to_numpy().astype(np.int64),
            pos_rows,
            neg_matrix,
            neg_len,
            rng,
        )
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "refit",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": float("nan"),
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(f"refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")

    torch.save(
        {
            "state_dict": final_model.state_dict(),
            "config": {
                "n_users": context.n_users,
                "n_items": context.n_items,
                "embed_dim": EMBED_DIM,
                "mlp_hidden": list(MLP_HIDDEN),
                "objective": LOSS,
                "negative_sampling": "per-epoch exposed negatives",
                "negative_count": int(CFG.negative_count),
                "negative_max": int(CFG.negative_count),
                "sampler_epoch_base": 0,
                "max_epochs": NUM_EPOCHS,
                "best_epoch": stopper.best_epoch,
                "epochs": stopper.best_epoch,
                "tune_ndcg": stopper.best_score,
                "ndcg_k": TUNE_NDCG_K,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "training_data": "labels_train full training pool",
                "training_groups": n_groups,
            },
        },
        config["artifact_dir"] / "model.pt",
    )
    pl.DataFrame(training_log).write_csv(config["artifact_dir"] / "training_log.csv")
    print("Saved model:", config["artifact_dir"] / "model.pt")
    TRAINED_MODELS[LOSS] = final_model

## 5. Generate the standard candidate table

`select_top_k` scores each request against articles available by its timestamp, removes the request's history, ranks by score descending, training exposures descending, then ascending `news_code`, and backfills short lists from popularity. Because the MLP is not a bilinear dot product, scores are evaluated in bounded request-article pair chunks rather than one matrix multiply. Requests with no history receive the global popularity order directly. Rows are streamed through `CandidateWriter` in timestamp order to `artifacts/ncf_<loss>/recommendations_<split>_top500.parquet`.


In [8]:
def select_top_k(scores, popularity, news_code, k, excluded=None, fallback_order=None):
    if fallback_order is not None:
        return fallback_order[fallback_order < scores.size][:k]
    available = np.ones(scores.size, dtype=bool)
    if excluded is not None:
        available[excluded] = False
    order = np.lexsort((news_code, -popularity, -scores))
    return order[available[order]][:k]

In [9]:
catalogue = pl.DataFrame(
    {
        "article_row": np.arange(1, context.n_items, dtype=np.int32),
        "news_code": context.news_code[1:],
        "available_from": context.available_from[1:],
        "popularity": context.popularity[1:],
    }
).sort(["available_from", "news_code"])

article_rows = catalogue["article_row"].to_numpy().astype(np.int64)
news_sorted = catalogue["news_code"].to_numpy().astype(np.int64)
available_sorted = catalogue["available_from"].to_numpy().astype("datetime64[ns]")
popularity_sorted = catalogue["popularity"].to_numpy().astype(np.float32)

pos_in_sorted = np.full(context.n_items, -1, dtype=np.int64)
pos_in_sorted[article_rows] = np.arange(article_rows.size, dtype=np.int64)
global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))

profiles = pl.read_parquet(
    sorted((FEATURES / f"request_profiles_{TARGET_SPLIT}").glob("*.parquet")),
    columns=["impression_uid", "user_code", "timestamp", "full_history_rows"],
).sort("timestamp", maintain_order=True)

impression_uids = profiles["impression_uid"].to_numpy()
target_users = profiles["user_code"].to_numpy().astype(np.int64)
timestamps = profiles["timestamp"].to_numpy().astype("datetime64[ns]")
history_rows = profiles["full_history_rows"].to_list()
starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
print(f"Scoring requests: {len(timestamps):,}")

Scoring requests: 31,624


In [10]:
def score_batch(model, user_vectors, item_vectors, max_pairs=SCORE_PAIRS):
    users_t = torch.from_numpy(user_vectors).to(DEVICE)
    items_t = torch.from_numpy(item_vectors).to(DEVICE)
    n_users, n_items = users_t.shape[0], items_t.shape[0]
    items_per_chunk = min(n_items, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    scores = np.empty((n_users, n_items), dtype=np.float32)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            user_block = users_t[u0:u1]
            for i0 in range(0, n_items, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items)
                rows, cols = u1 - u0, i1 - i0
                scores[u0:u1, i0:i1] = (
                    model.score_embeddings(
                        user_block[:, None, :].expand(rows, cols, -1),
                        items_t[i0:i1][None, :, :].expand(rows, cols, -1),
                    )
                    .cpu()
                    .numpy()
                )
    return scores

In [11]:
for loss in RUN_LOSSES:
    LOSS = loss
    config = LOSS_CONFIGS[loss]
    model = TRAINED_MODELS[loss]
    print(f"[{LOSS}] Scoring {config['model_name']}")
    model.eval()
    user_emb = model.user_embedding.weight.detach().cpu().numpy()[target_users]
    item_emb = model.item_embedding.weight.detach().cpu().numpy()[article_rows]
    writer = CandidateWriter(config["recommendations_path"])
    started = time.time()
    try:
        for batch_no, (start, end) in enumerate(zip(starts, ends), 1):
            timestamp = timestamps[start]
            n_eligible = int(np.searchsorted(available_sorted, timestamp, side="right"))
            scores = score_batch(model, user_emb[start:end], item_emb[:n_eligible])
            popularity_batch = popularity_sorted[:n_eligible]
            news_batch = news_sorted[:n_eligible]
            selected_batch = np.empty((end - start, CANDIDATE_CAP), dtype=np.int64)
            for local, profile_row in enumerate(range(start, end)):
                rows = history_rows[profile_row]
                excluded = None
                if rows:
                    excluded = pos_in_sorted[np.asarray(rows, dtype=np.int64)]
                    excluded = excluded[(excluded >= 0) & (excluded < n_eligible)]
                selected_batch[local] = select_top_k(
                    scores[local],
                    popularity_batch,
                    news_batch,
                    CANDIDATE_CAP,
                    excluded=excluded,
                    fallback_order=None if rows else global_popularity_order,
                )
            writer.add(
                np.repeat(impression_uids[start:end], CANDIDATE_CAP),
                np.tile(cutoff_values, end - start),
                news_batch[selected_batch.reshape(-1)],
            )
            if batch_no % 1000 == 0 or batch_no == len(starts):
                print(
                    f"Scored {batch_no:,}/{len(starts):,} timestamp batches; "
                    f"{writer.rows:,} rows",
                    flush=True,
                )
    finally:
        writer.close()

    print(f"Wrote {writer.rows:,} candidate rows in {time.time() - started:.1f}s")
    print("Candidates:", config["recommendations_path"])

[bpr] Scoring ncf_bpr


Scored 1,000/24,580 timestamp batches; 523,500 rows
Scored 2,000/24,580 timestamp batches; 1,099,000 rows
Scored 3,000/24,580 timestamp batches; 1,718,000 rows
Scored 4,000/24,580 timestamp batches; 2,367,000 rows
Scored 5,000/24,580 timestamp batches; 3,048,000 rows
Scored 6,000/24,580 timestamp batches; 3,734,000 rows
Scored 7,000/24,580 timestamp batches; 4,404,000 rows
Scored 8,000/24,580 timestamp batches; 5,095,500 rows
Scored 9,000/24,580 timestamp batches; 5,771,000 rows
Scored 10,000/24,580 timestamp batches; 6,465,500 rows
Scored 11,000/24,580 timestamp batches; 7,188,000 rows
Scored 12,000/24,580 timestamp batches; 7,901,000 rows
Scored 13,000/24,580 timestamp batches; 8,620,500 rows
Scored 14,000/24,580 timestamp batches; 9,331,000 rows
Scored 15,000/24,580 timestamp batches; 10,035,500 rows
Scored 16,000/24,580 timestamp batches; 10,730,000 rows
Scored 17,000/24,580 timestamp batches; 11,417,000 rows
Scored 18,000/24,580 timestamp batches; 12,063,000 rows
Scored 19,000/24,

## 6. Evaluation

Uses the shared `CandidateEvaluator` and writes `results/candidate_generation_<model>_<split>.csv`. It reports recall (pooled and impression-mean), hit rate, long-tail recall, cold-article recall, novelty (bits), catalogue coverage and personalisation at each M.


In [12]:
from cg_evaluation import CandidateEvaluator

for loss in RUN_LOSSES:
    config = LOSS_CONFIGS[loss]
    evaluator = CandidateEvaluator(root=DATA_ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate_to_csv(
        config["recommendations_path"],
        model_name=config["model_name"],
        m_values=M_VALUES,
        output_dir=RESULTS,
    )
    print(f"[{loss}]")
    print(evaluation.sort("m"))
    print("Results:", config["result_path"])

[bpr]
shape: (32, 7)
┌─────────┬────────────┬─────┬────────────────────────┬───────────┬────────────┬─────────────┐
│ model   ┆ split      ┆ m   ┆ metric                 ┆ value     ┆ numerator  ┆ denominator │
│ ---     ┆ ---        ┆ --- ┆ ---                    ┆ ---       ┆ ---        ┆ ---         │
│ str     ┆ str        ┆ i64 ┆ str                    ┆ f64       ┆ f64        ┆ f64         │
╞═════════╪════════════╪═════╪════════════════════════╪═══════════╪════════════╪═════════════╡
│ ncf_bpr ┆ validation ┆ 50  ┆ catalogue_coverage     ┆ 0.002209  ┆ 105.0      ┆ 47538.0     │
│ ncf_bpr ┆ validation ┆ 50  ┆ cold_article_recall    ┆ 0.0       ┆ 0.0        ┆ 342.0       │
│ ncf_bpr ┆ validation ┆ 50  ┆ hit_rate               ┆ 0.014546  ┆ 460.0      ┆ 31624.0     │
│ ncf_bpr ┆ validation ┆ 50  ┆ long_tail_recall       ┆ 0.009716  ┆ 95.0       ┆ 9778.0      │
│ ncf_bpr ┆ validation ┆ 50  ┆ novelty_bits           ┆ 15.509738 ┆ 2.4524e7   ┆ 1.5812e6    │
│ …       ┆ …          ┆ …   

## Notes and limitations

- The user representation is a static `user_code` embedding, so two requests from the same user receive the same ranking apart from availability and history exclusion.
- Positives are session clicks and negatives are observed non-clicks from the same slate, so the model inherits the logging system's exposure and position bias.
- Up to four exposed negatives are redrawn from the same slate each epoch, which controls popularity bias better than a fixed negative sample.
- The MLP adds capacity over GMF but makes full scoring quadratic in requests and pairs: every request-article pair needs a forward pass, so candidate generation is much slower than a dot-product model.
- The model is fitted once on the training split and never refreshed, so it cannot react to drift inside validation or test.
- The MLP output is a ranking score, not a calibrated click probability.
- Articles without training interactions carry untrained embeddings, so their scores are effectively arbitrary and `cold_article_recall` depends mainly on the popularity fallback.
- Requests with empty history receive the global popularity order and are therefore identical to the popularity control.
- `available_from` is the project-wide observable-availability proxy, not a verified publication timestamp.
